# Fixture context — double vs single gameweek (`total_points`)

*Read-only informative artifact. This notebook characterises how
double-gameweek (DGW) context changes the target picture so a human can decide
how to read it. It produces no gate decisions and no PROCEED/STOP verdict.*

## Questions a manager asks about double gameweeks

- **Do players score more in a double gameweek?** A DGW row sums two fixtures
  into one `total_points`, so of course the raw total is higher — but how much?
- **Is the DGW bump just mechanical, or do players perform better per
  fixture?** If a DGW is worth roughly twice a single gameweek, the gain is
  pure fixture-doubling; if it is worth *more* than twice per fixture, players
  genuinely lift; if *less*, fatigue/rotation bites.
- **Does the effect differ by position?** Does doubling reward attackers more
  than defenders, or is it even?

This is the layer the `structure/` and `temporal/` notebooks deferred to: they
pooled SGW and DGW rows and flagged the fixture-doubling confound; **here** it
is examined head-on. Everything below is **season-pooled** over the study
range.

## Setup

Load the mart, restrict to the **whole season** (GW 1 to the latest completed
GW) and the **participation** population (`minutes > 0`), then split featured
player-gameweeks by `fixture_context` into **SGW** and **DGW** cohorts.

This is a *descriptive characterisation* notebook, so it uses the full season —
no early-GW lower bound (that GW-6 cut in the older EDA-1 record was a
*predictive-evaluation* choice, not relevant here).

The population is everyone who **actually featured**: available players with
`minutes > 0`. This is a **participation** filter (the player appeared), **not
a performance gate**. `minutes` can be NULL for some rows; `minutes > 0`
naturally excludes those. The 60-minute performance-boundary question is
**deferred to the `population/` layer** and is deliberately not baked in here.

**Blank-gameweek (BGW) note.** The mart's `fixture_context` also carries a
`BGW` value, but BGW rows have NULL `minutes` (the player had no fixture), so
the `minutes > 0` participation filter already excludes them. Only `SGW` and
`DGW` rows survive into the cohorts below.

**Thin-DGW caveat.** Double-gameweeks are rare. Pre-filter there are ~171
DGW player-gameweeks against ~11k SGW rows, and fewer still survive position
splits. Every DGW statistic below is **small-n** and should be read
cautiously; the actual n per position is stated in each table.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

from dal.pipeline import load as load_mart, run as run_pipeline
from dal.exceptions import MartNotBuiltError, MartSchemaError
from research.kernels.descriptive.distribution import (
    compute_distribution_stats,
    compare_cohorts,
)

try:
    _result = load_mart()
except (MartNotBuiltError, MartSchemaError) as _e:
    print(f"Rebuilding mart ({type(_e).__name__})...")
    run_pipeline(force=True)
    _result = load_mart()

# Descriptive characterisation uses the WHOLE season: GW 1 to the latest
# completed GW. No early-GW lower bound (that was a predictive-evaluation
# choice in the older EDA-1 record, not relevant here).
STUDY_GW_MIN = 1
STUDY_GW_MAX = _result.data_cutoff_gw

# Analytical population: PARTICIPATION filter, not a performance gate.
# Available players who actually featured -> minutes > 0. `minutes` can be NULL
# for some rows; minutes > 0 naturally excludes NULLs (NULL comparisons are
# False). The 60-minute performance boundary is NOT imposed here -- that
# question is deferred to the population/ layer. BGW rows have NULL minutes and
# are excluded by this same filter.
mart = _result.mart
df = mart[mart["gw"].between(STUDY_GW_MIN, STUDY_GW_MAX)].copy()
df = df[df["minutes"].notna() & (df["minutes"] > 0)].copy()

POSITIONS = ["GK", "DEF", "MID", "FWD"]
CONTEXTS = ["SGW", "DGW"]

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", "{:.3f}".format)

print(f"Study range: GW {STUDY_GW_MIN} - GW {STUDY_GW_MAX} (whole season, from mart data_cutoff_gw)")
print(f"Population: minutes > 0 (participation, not a performance gate), n = {len(df):,} player-gameweeks")
print("\nfixture_context counts (after minutes > 0 filter):")
for ctx in CONTEXTS:
    print(f"  {ctx}: {int((df['fixture_context'] == ctx).sum()):>6,}")

## DGW prevalence — n per position

**What we measure** — the count of featured player-gameweeks in each
`fixture_context` (SGW / DGW), broken down by position. This is the sample-size
table every comparison below rests on.

**What it means** — it states plainly how thin the DGW cohort is at each
position, so the reader can weight the SGW-vs-DGW comparisons accordingly. A
position with only a handful of DGW rows cannot support a confident read.

**What it doesn't mean** — counts say nothing about scoring; they only size the
cohorts. A small DGW n does not make the comparison wrong, only uncertain.

**Guiding question** — *How many featured rows back the DGW comparison at each
position?*

In [ ]:
# n per (position, context) — sizes the cohorts for every comparison below.
_counts = df.groupby(["position", "fixture_context"]).size().unstack(fill_value=0).reset_index()
_all = pd.DataFrame([{"position": "ALL", **df["fixture_context"].value_counts().to_dict()}])
ctx_counts = pd.concat([_counts, _all], ignore_index=True)
for col in ["SGW", "DGW", "BGW"]:
    if col not in ctx_counts.columns: ctx_counts[col] = 0
ctx_counts["DGW_share_%"] = (ctx_counts["DGW"] / (ctx_counts["SGW"] + ctx_counts["DGW"]).replace(0, float("nan")) * 100).round(2)
display(ctx_counts[["position", "SGW", "DGW", "DGW_share_%"]])


## (a) `total_points` distribution — SGW vs DGW by position (raw totals)

**What we measure** — the full distribution of `total_points` (count, mean,
median, percentiles, max) for the SGW and DGW cohorts, **within each
position**, via `compare_cohorts` with cohorts = {SGW, DGW}.

*Stat glossary for this table:* **mean** — arithmetic average, pulled up by rare hauls; **median** — middle value, the robust "typical" return; **std** — typical distance from the mean in points, sensitive to outliers; **IQR (p75−p25)** — spread of the middle 50%, robust to skew; **p90/p99** — score exceeded by only 10%/1% of appearances (practical ceiling); **skew** — positive = long right tail of rare high scores.

**What it means** — this is the raw answer to "do players score more in a
double gameweek?" for squad-planning around a DGW: how much more total a DGW
player banks. **Crucially, a DGW row sums TWO fixtures into one
`total_points`** — so a higher DGW mean is *expected* and is the **mechanical
fixture-doubling** that the `structure/` and `temporal/` layers deliberately
left for this layer to examine. **This is that examination.** Read the raw gap
here as "what the doubled total looks like", then read section (b) to separate
mechanical doubling from genuine per-fixture lift.

**What it doesn't mean** — the raw DGW total is **not** a per-fixture
performance: a DGW player who returns ~2x an SGW player may simply have played
twice at the same level. This table cannot, on its own, tell mechanical
doubling from real outperformance — that is exactly what (b) addresses. All
figures are **season-pooled**, and the DGW cohort is **small-n** (see the
counts table) so the tails especially are noisy.

**Guiding question** — *How much more do players score in raw total during a
double gameweek, and does the raw gap differ by position?*

In [ ]:
CONTEXTS = ["SGW", "DGW"]
raw_stats = (
    df[df["fixture_context"].isin(CONTEXTS)]
    .groupby(["position", "fixture_context"])["total_points"]
    .agg(["count", "mean", "median",
          lambda s: s.quantile(0.9), lambda s: s.max()])
    .rename(columns={"<lambda_0>": "p90", "<lambda_1>": "max"})
    .reset_index()
    .round(2)
)
display(raw_stats)


## (b) Per-fixture view — `total_points / fixture_count` SGW vs DGW by position

**What we measure** — the same SGW-vs-DGW distribution comparison, but on
**points-per-fixture** (`total_points / fixture_count`): SGW rows divide by 1,
DGW rows divide by 2. Again via `compare_cohorts` with cohorts = {SGW, DGW},
within each position.

*Stat glossary for this table:* **mean** — arithmetic average, pulled up by rare hauls; **median** — middle value, the robust "typical" return; **std** — typical distance from the mean in points, sensitive to outliers; **IQR (p75−p25)** — spread of the middle 50%, robust to skew; **p90/p99** — score exceeded by only 10%/1% of appearances (practical ceiling); **skew** — positive = long right tail of rare high scores.

**What it means** — this normalisation strips out the mechanical doubling so
the reader can see whether DGW outperformance is **purely mechanical** or
**real**:

- DGW per-fixture mean ~= SGW mean  -> the raw DGW bump was **purely
  mechanical** (two ordinary matches stacked, no per-match lift);
- DGW per-fixture mean **>** SGW mean -> players genuinely **lift** per fixture
  in a DGW (e.g. easier opponents, in-form sides);
- DGW per-fixture mean **<** SGW mean -> per-fixture output **dips** (rotation,
  fatigue, tougher second fixture).

Per-fixture is shown here as a **descriptive normalisation FOR COMPARISON** — a
lens to separate mechanism from performance — **not** a mandated treatment of
the target. Nothing downstream is required to divide by `fixture_count`.

**What it doesn't mean** — dividing by `fixture_count` averages the two
fixtures into one number; it hides which of the two fixtures carried the haul
and discards the within-DGW split. It is also **season-pooled** and rests on
the same **small-n** DGW cohort, so a per-fixture gap of a point or two is
within noise. It is not a forecast and not a recommendation to roster per
fixture.

**Guiding question** — *Once we divide out the second fixture, do DGW players
still outperform per fixture — is the bump real or just mechanical?*

In [ ]:
df_ppf = df.copy()
df_ppf["points_per_fixture"] = df_ppf["total_points"] / df_ppf["fixture_count"]

ppf_rows = []
for pos in POSITIONS:
    sub = df_ppf[df_ppf.position == pos]
    cohorts = {ctx: sub[sub["fixture_context"] == ctx] for ctx in CONTEXTS}
    stats = compare_cohorts(cohorts, value_col="points_per_fixture")
    for ctx in CONTEXTS:
        s = stats.loc[ctx]
        ppf_rows.append({
            "position": pos,
            "context": ctx,
            "n": int(s["count"]) if not np.isnan(s["count"]) else 0,
            "ppf_mean": s["mean"],
            "ppf_median": s["median"],
            "ppf_std": s["std"],
        })
ppf_by_pos = pd.DataFrame(ppf_rows)
display(ppf_by_pos.round(2))

In [ ]:
# Side-by-side bars reveal what the two tables hide: the RAW mean roughly
# doubles for DGW (mechanical), while the PER-FIXTURE mean sits much closer to
# SGW -- the gap between those two panels IS the mechanical-vs-real story.
colours = {"SGW": "#1f77b4", "DGW": "#d62728"}
x = np.arange(len(POSITIONS))
width = 0.38
fig, (ax_raw, ax_ppf) = plt.subplots(1, 2, figsize=(13, 4), sharex=True)

raw_mean = raw_by_pos.pivot(index="position", columns="context", values="mean").reindex(POSITIONS)
ppf_mean = ppf_by_pos.pivot(index="position", columns="context", values="ppf_mean").reindex(POSITIONS)

for ax, data, title in [
    (ax_raw, raw_mean, "RAW total_points mean (DGW carries 2 fixtures)"),
    (ax_ppf, ppf_mean, "PER-FIXTURE mean (total_points / fixture_count)"),
]:
    for i, ctx in enumerate(CONTEXTS):
        ax.bar(x + (i - 0.5) * width, data[ctx].to_numpy(dtype=float),
               width, label=ctx, color=colours[ctx])
    ax.set_xticks(x)
    ax.set_xticklabels(POSITIONS)
    ax.set_title(title, fontsize=10)
    ax.set_ylabel("mean total_points")
    ax.legend()
fig.suptitle("SGW vs DGW: raw doubling vs per-fixture lift (minutes > 0, small-n DGW)", y=1.02)
plt.tight_layout()
plt.show()

## What double gameweeks look like

Plain-language summary of the descriptive picture (not a verdict):

- The **raw** `total_points` table (a) shows DGW rows sitting well above SGW
  rows — as expected, because a DGW row **sums two fixtures**. This is the
  mechanical fixture-doubling the `structure/` and `temporal/` layers deferred
  here, now made explicit.
- The **per-fixture** table (b) divides that doubling out. Comparing the DGW
  per-fixture mean to the SGW mean shows whether the bump is **purely
  mechanical** (per-fixture ~= SGW), a **real lift** (per-fixture > SGW), or a
  **dip** (per-fixture < SGW) — position by position.
- Any read here is constrained by **small-n**: the DGW cohort is a few dozen
  rows per position (see the counts table), so per-fixture gaps of a point or
  two are within noise and the tails are unstable.

All figures are **whole-season**, season-pooled, over the **participation**
population (`minutes > 0` — the player appeared — not a performance gate; the
60-minute boundary is deferred to the `population/` layer). Per-fixture
normalisation is shown **for comparison only**, not mandated. FDR (fixture
difficulty) and home/away are treated in the sibling notebooks.